In [ ]:
# %% [markdown]
# # GeoSDOH & Diabetes Longitudinal Study Notebook
# ### ADI, HbA1c, and Staggered Design (t0 = First Uncontrolled)
#
# This notebook:
# 1. Loads EMR (HbA1c) data with ADI linkage
# 2. Cleans and standardizes key variables
# 3. Defines t0 (first uncontrolled HbA1c > 7%)
# 4. Builds baseline features (12-month lookback)
# 5. Constructs 6-month, 12-month, 24-month outcomes
# 6. Draws a stratified probability sample from the full t0 cohort
# 7. Compares full vs sampled distributions (representativeness checks)
# 8. Applies design weights and optional IPW
# 9. Fits weighted logistic models with cluster-robust SE
# 10. Demonstrates a simple Markov transition model
# 11. Sketches a Cox PH survival model for time to regaining control

# %% [markdown]
# ## 0. Imports & Config

# %%
import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.utils import resample

import statsmodels.api as sm

import geopandas as gpd

# Optional (for survival)
try:
    from lifelines import CoxPHFitter
except ImportError:
    CoxPHFitter = None

import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings("ignore")

# %% [markdown]
# ## 1. Load Data & Harmonize Column Names
#
# We assume your CSV has columns like:
# - ID
# - PATIENT ID / PATIENT
# - RESULT DATETIME, RESULT VALUE NUM, RESULT UNIT
# - ENCOUNTER DATE, AGE AT VISIT, SEX
# - HF, CKD, STRKE, ATHSCLTC, STEMI (+ *_DT variants)
# - ETHNICITY ORIG, RACE ORIG
# - CENSUS 2020 BLOCK ID / BLOCK GROUP ID
# - ADI NATIONAL RANK 22, ADI STATE RANK 22
# - RURAL COUNTY CODE DESCRIPTION
# - CENTER, DEPARTMENT NAME, COMBINED VITAL STATUS, CONSENTED YN
#
# Adjust the rename_map if needed.

# %%
EMR_PATH = "emr_raw.csv"

df_raw = pd.read_csv(EMR_PATH)

print("Raw columns:")
print(df_raw.columns.tolist())

# Map verbose column names to easier snake_case
rename_map = {
    "PATIENT ID": "patient_id",
    "PATIENT": "patient_id",
    "RESULT DATETIME": "result_datetime",
    "RESULT VALUE NUM": "hba1c_value",
    "RESULT UNIT": "result_unit",
    "ENCOUNTER DATE": "encounter_date",
    "AGE AT VISIT": "age",
    "SEX": "sex",
    "HF": "hf",
    "HF DT": "hf_dt",
    "CKD": "ckd",
    "CKD DT": "ckd_dt",
    "STRKE": "stroke",
    "STRKE DT": "stroke_dt",
    "ATHSCLTC": "athscltc",
    "ATHSCLTC DT": "athscltc_dt",
    "STEMI": "stemi",
    "STEMI DT": "stemi_dt",
    "ETHNICITY ORIG": "ethnicity",
    "RACE ORIG": "race",
    "CENSUS 2020 BLOCK GROUP ID": "block_group_id",
    "CENSUS 2020 BLOCK ID": "block_id",
    "ADI NATIONAL RANK 22": "adi_natrank",
    "ADI STATE RANK 22": "adi_state_rank",
    "RURAL COUNTY CODE DESCRIPTION": "rural_desc",
    "DEPARTMENT NAME": "department",
    "CENTER": "center",
    "COMBINED VITAL STATUS": "vital_status",
    "CONSENTED YN": "consented"
}

df = df_raw.rename(columns={k: v for k, v in rename_map.items() if k in df_raw.columns})

# %% [markdown]
# ### 1.1 Basic Cleaning

# %%
# Keep only adults (age >= 18)
df = df[df["age"] >= 18]

# Parse dates
for col in ["result_datetime", "encounter_date", "hf_dt", "ckd_dt",
            "stroke_dt", "athscltc_dt", "stemi_dt"]:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

# Sort by patient & encounter time
df = df.sort_values(["patient_id", "encounter_date", "result_datetime"])

df.head()

# %% [markdown]
# ### 1.2 Filter to HbA1c Labs
#
# If you have a specific HbA1c PROC CODE or test name, filter there.
# For now we assume all rows with `hba1c_value` are HbA1c labs.

# %%
# Remove rows without HbA1c value
df = df.dropna(subset=["hba1c_value"])

# Optionally restrict to a specific unit (e.g., %)
if "result_unit" in df.columns:
    df = df[df["result_unit"].astype(str).str.contains("%", case=False, na=False)]

# If encounter_date missing, fill from result_datetime
df["encounter_date"] = pd.to_datetime(df["encounter_date"], errors="coerce")
mask_missing_enc = df["encounter_date"].isna() & df["result_datetime"].notna()
df.loc[mask_missing_enc, "encounter_date"] = df.loc[mask_missing_enc, "result_datetime"]

df = df.dropna(subset=["patient_id", "encounter_date"])

df.head()

# %% [markdown]
# ## 2. Define Control Status and First Uncontrolled Time (t0)

# %%
df["uncontrolled"] = (df["hba1c_value"] > 7.0).astype(int)

# First date of uncontrolled (t0) for each patient
first_uncontrolled = (
    df[df["uncontrolled"] == 1]
    .groupby("patient_id")["encounter_date"]
    .min()
    .reset_index()
    .rename(columns={"encounter_date": "t0"})
)

df = df.merge(first_uncontrolled, on="patient_id", how="left")

# 12-month baseline window before t0
df["in_baseline_window"] = (
    df["t0"].notna()
    & (df["encounter_date"] >= df["t0"] - pd.Timedelta(days=365))
    & (df["encounter_date"] < df["t0"])
)

df.head()

# %% [markdown]
# ## 3. Build Baseline Features (One Row per Patient)

# %%
baseline = (
    df[df["in_baseline_window"]]
    .sort_values(["patient_id", "encounter_date"])
    .groupby("patient_id")
    .tail(1)  # last baseline encounter
    .copy()
)

# Baseline HbA1c
baseline["baseline_hba1c"] = baseline["hba1c_value"]

# ADI as numeric
baseline["adi_natrank"] = pd.to_numeric(baseline["adi_natrank"], errors="coerce")

# ADI quintiles
baseline["adi_quintile"] = pd.qcut(
    baseline["adi_natrank"], 5, labels=False, duplicates="drop"
) + 1

# Comorbidity flags
for c in ["hf", "ckd", "stroke", "athscltc", "stemi"]:
    if c in baseline.columns:
        baseline[c] = baseline[c].fillna(0)

baseline["age"] = baseline["age"].astype(float)
baseline["sex"] = baseline["sex"].astype(str).str.upper()

if "block_group_id" in baseline.columns:
    baseline["block_group_id"] = baseline["block_group_id"].astype(str)

baseline.head()

# %% [markdown]
# ### 3.1 HbA1c Slope in 12-Month Baseline Window

# %%
def compute_baseline_slope(df_all, patient_id, t0):
    if pd.isna(t0):
        return np.nan
    mask = (
        (df_all["patient_id"] == patient_id)
        & (df_all["encounter_date"] >= t0 - pd.Timedelta(days=365))
        & (df_all["encounter_date"] < t0)
    )
    tmp = df_all.loc[mask, ["encounter_date", "hba1c_value"]].dropna()
    if len(tmp) < 2:
        return np.nan
    tmp = tmp.sort_values("encounter_date")
    x = (tmp["encounter_date"] - tmp["encounter_date"].min()).dt.days.values
    y = tmp["hba1c_value"].values
    slope = np.polyfit(x, y, 1)[0]  # per day
    return slope * 365.0  # per year

baseline["hba1c_slope"] = baseline.apply(
    lambda r: compute_baseline_slope(df, r["patient_id"], r["t0"]), axis=1
)

baseline.head()

# %% [markdown]
# ## 4. Longitudinal Outcomes at 6, 12, 24 Months

# %%
def future_uncontrolled(df_all, pid, t0, months, window=3):
    if pd.isna(t0):
        return np.nan
    center = t0 + pd.DateOffset(months=months)
    lower = center - pd.DateOffset(months=window)
    upper = center + pd.DateOffset(months=window)
    tmp = df_all[
        (df_all["patient_id"] == pid)
        & (df_all["encounter_date"] >= lower)
        & (df_all["encounter_date"] <= upper)
    ].sort_values("encounter_date")
    if tmp.empty:
        return np.nan
    return int(tmp["hba1c_value"].iloc[-1] > 7.0)

baseline["out_6mo"] = baseline.apply(
    lambda r: future_uncontrolled(df, r["patient_id"], r["t0"], months=6, window=3),
    axis=1
)
baseline["out_1yr"] = baseline.apply(
    lambda r: future_uncontrolled(df, r["patient_id"], r["t0"], months=12, window=3),
    axis=1
)
baseline["out_2yr"] = baseline.apply(
    lambda r: future_uncontrolled(df, r["patient_id"], r["t0"], months=24, window=6),
    axis=1
)

baseline.head()

# %%
cohort6  = baseline.dropna(subset=["out_6mo"]).copy()
cohort12 = baseline.dropna(subset=["out_1yr"]).copy()
cohort24 = baseline.dropna(subset=["out_2yr"]).copy()

print("N 6mo:", len(cohort6))
print("N 1yr:", len(cohort12))
print("N 2yr:", len(cohort24))

# %% [markdown]
# ## 5. Stratified Sampling for a Smaller, Representative Cohort
#
# We now build a **stratified probability sample** from the full t₀ cohort, then restrict our horizon cohorts to that sample.

# %% [markdown]
# ### 5.1 Define Full t₀ Cohort & Strata

# %%
# Full t0 cohort: patients with t0 and baseline row
full_cohort = baseline.dropna(subset=["adi_natrank"]).copy()
full_cohort["t0_year"] = full_cohort["t0"].dt.year

# Rural flag
if "rural_desc" in full_cohort.columns:
    full_cohort["rural_flag"] = np.where(
        full_cohort["rural_desc"].astype(str).str.contains("Rural", case=False, na=False),
        1, 0
    )
else:
    full_cohort["rural_flag"] = 0

# Clean sex categories
full_cohort["sex"] = full_cohort["sex"].astype(str).str.upper()
full_cohort.loc[~full_cohort["sex"].isin(["M", "F"]), "sex"] = "OTHER"

print("Full t0 cohort size:", len(full_cohort))

# %% [markdown]
# ### 5.2 Proportional Stratified Sample with Design Weights

# %%
N_target = 20000  # desired sample size (adjust as needed)

strat_cols = ["adi_quintile", "sex", "t0_year", "rural_flag"]

# Count full cohort per stratum
stratum_counts = (
    full_cohort
    .groupby(strat_cols)
    .size()
    .reset_index(name="N_full")
)

total_full = stratum_counts["N_full"].sum()
stratum_counts["prop_full"] = stratum_counts["N_full"] / total_full

# Proportional allocation
stratum_counts["N_sample"] = (stratum_counts["prop_full"] * N_target).round().astype(int)
stratum_counts.loc[(stratum_counts["N_full"] > 0) & (stratum_counts["N_sample"] == 0), "N_sample"] = 1

stratum_counts.head()

# %%
# Merge planned sample sizes back onto full_cohort
full_cohort_with_n = full_cohort.merge(
    stratum_counts,
    on=strat_cols,
    how="left"
)

# Draw sample within each stratum
sampled_rows = []

for _, row in stratum_counts.iterrows():
    mask = (
        (full_cohort_with_n["adi_quintile"] == row["adi_quintile"]) &
        (full_cohort_with_n["sex"] == row["sex"]) &
        (full_cohort_with_n["t0_year"] == row["t0_year"]) &
        (full_cohort_with_n["rural_flag"] == row["rural_flag"])
    )
    df_stratum = full_cohort_with_n[mask]
    n_full = int(row["N_full"])
    n_sample = int(row["N_sample"])

    if n_full == 0 or n_sample == 0:
        continue

    if n_sample >= n_full:
        df_samp = df_stratum
    else:
        df_samp = df_stratum.sample(n=n_sample, random_state=42)

    sampled_rows.append(df_samp)

sampled_cohort = pd.concat(sampled_rows, ignore_index=True)
print("Sampled cohort size:", len(sampled_cohort))

# %%
# Compute actual sampling fraction and design weights
sample_counts = (
    sampled_cohort
    .groupby(strat_cols)
    .size()
    .reset_index(name="N_sample_actual")
)

sampled_cohort = sampled_cohort.merge(
    sample_counts,
    on=strat_cols,
    how="left"
)

sampled_cohort["sampling_frac"] = sampled_cohort["N_sample_actual"] / sampled_cohort["N_full"]
sampled_cohort["weight_design"] = 1.0 / sampled_cohort["sampling_frac"].replace(0, np.nan)

sampled_cohort[["patient_id", "adi_quintile", "sex", "t0_year", "rural_flag", "weight_design"]].head()

# %% [markdown]
# ### 5.3 Representativeness Checks: Full vs Sampled Distributions
#
# We compare distributions for:
# - ADI
# - age
# - sex
# - t0_year
# - rural_flag

# %%
def compare_dist_full_sample(full, sample, col, bins=20, is_categorical=False):
    print(f"\n=== {col} ===")

    if is_categorical:
        full_dist = full[col].value_counts(normalize=True).sort_index()
        samp_dist = sample[col].value_counts(normalize=True).sort_index()
        comp = pd.concat(
            [full_dist.rename("full"), samp_dist.rename("sample")],
            axis=1
        ).fillna(0)
        print(comp)

        ax = comp.plot(kind="bar", figsize=(6, 4))
        ax.set_title(f"Distribution of {col}: Full vs Sample")
        ax.set_ylabel("Proportion")
        ax.legend(["Full cohort", "Sampled cohort"])
        plt.tight_layout()
        plt.show()

    else:
        fig, axes = plt.subplots(1, 2, figsize=(10, 4))
        full[col].dropna().hist(ax=axes[0], bins=bins)
        axes[0].set_title(f"Full cohort {col}")
        samp = sample[col].dropna()
        samp.hist(ax=axes[1], bins=bins)
        axes[1].set_title(f"Sampled cohort {col}")
        plt.tight_layout()
        plt.show()

# Compare ADI, age
compare_dist_full_sample(full_cohort, sampled_cohort, "adi_natrank", bins=20, is_categorical=False)
compare_dist_full_sample(full_cohort, sampled_cohort, "age", bins=20, is_categorical=False)

# Compare sex, t0_year, rural_flag
compare_dist_full_sample(full_cohort, sampled_cohort, "sex", is_categorical=True)
compare_dist_full_sample(full_cohort, sampled_cohort, "t0_year", is_categorical=True)
compare_dist_full_sample(full_cohort, sampled_cohort, "rural_flag", is_categorical=True)

# %% [markdown]
# ## 6. Restrict Horizon Cohorts to Sampled Patients

# %%
cohort6_small = cohort6.merge(
    sampled_cohort[["patient_id", "weight_design"]],
    on="patient_id",
    how="inner"
)
cohort12_small = cohort12.merge(
    sampled_cohort[["patient_id", "weight_design"]],
    on="patient_id",
    how="inner"
)
cohort24_small = cohort24.merge(
    sampled_cohort[["patient_id", "weight_design"]],
    on="patient_id",
    how="inner"
)

print("Sampled N 6mo:", len(cohort6_small))
print("Sampled N 1yr:", len(cohort12_small))
print("Sampled N 2yr:", len(cohort24_small))

# %% [markdown]
# ## 7. IPW for Outcome Availability (Optional)

# %%
def ipw_weights(full_df, cohort_df, outcome_col):
    full = full_df.copy()
    full["has_outcome"] = ~full[outcome_col].isna()

    features = ["age", "baseline_hba1c", "adi_natrank"]
    for f in features:
        if f not in full.columns:
            full[f] = np.nan

    X_full = full[features].fillna(full[features].median())
    y_full = full["has_outcome"].astype(int)

    lr = LogisticRegression(max_iter=1000)
    lr.fit(X_full, y_full)

    p_hat = lr.predict_proba(X_full)[:, 1]
    full["p_hat"] = p_hat

    cohort = cohort_df.merge(
        full[["patient_id", "p_hat"]],
        on="patient_id",
        how="left"
    )

    cohort["ipw"] = 1.0 / cohort["p_hat"].clip(lower=1e-3)
    return cohort

cohort6_small  = ipw_weights(baseline, cohort6_small,  "out_6mo")
cohort12_small = ipw_weights(baseline, cohort12_small, "out_1yr")
cohort24_small = ipw_weights(baseline, cohort24_small, "out_2yr")

cohort6_small[["patient_id", "weight_design", "ipw"]].head()

# %% [markdown]
# ## 8. Weighted Logistic Regression with Cluster-Robust SE
#
# We now fit models using:
# - Outcome: out_6mo / out_1yr / out_2yr
# - Predictors: ADI, age, baseline HbA1c, slope, comorbidities, sex
# - Weights: design * IPW
# - Cluster: block group (or center/patient if block_group_id missing)

# %%
def fit_weighted_logit(df_in, outcome_col, cluster_col=None):
    df_mod = df_in.copy()

    predictors = [
        "adi_natrank", "age", "baseline_hba1c", "hba1c_slope",
        "hf", "ckd", "stroke", "athscltc", "stemi"
    ]
    predictors = [p for p in predictors if p in df_mod.columns]

    df_mod["sex_female"] = np.where(df_mod["sex"].str.upper().str.startswith("F"), 1, 0)
    predictors.append("sex_female")

    X = df_mod[predictors].fillna(df_mod[predictors].median())
    y = df_mod[outcome_col].astype(int)

    X = sm.add_constant(X)

    df_mod["final_weight"] = df_mod.get("weight_design", 1.0) * df_mod.get("ipw", 1.0)

    if cluster_col and (cluster_col in df_mod.columns):
        groups = df_mod[cluster_col]
    else:
        groups = df_mod["patient_id"]

    model = sm.GLM(y, X, family=sm.families.Binomial(), freq_weights=df_mod["final_weight"])
    res = model.fit()

    res_robust = res.get_robustcov_results(cov_type="cluster", groups=groups)

    return res_robust, predictors

res6_small, preds6 = fit_weighted_logit(cohort6_small,  "out_6mo", cluster_col="block_group_id")
res12_small, preds12 = fit_weighted_logit(cohort12_small, "out_1yr", cluster_col="block_group_id")
res24_small, preds24 = fit_weighted_logit(cohort24_small, "out_2yr", cluster_col="block_group_id")

print("6-month model summary:")
print(res6_small.summary())

# %%
print("12-month model summary:")
print(res12_small.summary())

# %%
print("24-month model summary:")
print(res24_small.summary())

# %% [markdown]
# ## 9. Markov-Style Transition Model (Controlled ↔ Uncontrolled)

# %%
df_markov = df[df["t0"].notna()].copy()
df_markov = df_markov.sort_values(["patient_id", "encounter_date"])
df_markov["state"] = df_markov["uncontrolled"].astype(int)

transitions = np.zeros((2, 2))
for pid, group in df_markov.groupby("patient_id"):
    s = group["state"].values
    for i in range(len(s) - 1):
        transitions[s[i], s[i+1]] += 1

row_sums = transitions.sum(axis=1, keepdims=True)
transition_probs = transitions / np.where(row_sums == 0, 1, row_sums)

print("Transition counts:\n", transitions)
print("Transition probabilities:\n", transition_probs)
print("P(controlled→uncontrolled):", transition_probs[0, 1])
print("P(uncontrolled→controlled):", transition_probs[1, 0])

# %% [markdown]
# ## 10. Cox PH Model for Time to Regain Control (Sketch)

# %%
if CoxPHFitter is not None:
    records = []
    for pid, group in df[df["t0"].notna()].groupby("patient_id"):
        group = group.sort_values("encounter_date")
        t0 = group["t0"].iloc[0]
        post = group[group["encounter_date"] > t0]
        if post.empty:
            continue

        time_to_event = None
        event = 0
        for _, row in post.iterrows():
            if row["hba1c_value"] <= 7.0:
                time_to_event = (row["encounter_date"] - t0).days
                event = 1
                break

        if time_to_event is None:
            time_to_event = (post["encounter_date"].max() - t0).days
            event = 0

        base_row = baseline[baseline["patient_id"] == pid].head(1)
        if base_row.empty:
            continue

        rec = {
            "patient_id": pid,
            "time": time_to_event,
            "event": event,
            "adi_natrank": base_row["adi_natrank"].iloc[0],
            "age": base_row["age"].iloc[0],
            "baseline_hba1c": base_row["baseline_hba1c"].iloc[0]
        }
        records.append(rec)

    surv_df = pd.DataFrame(records).dropna()
    cph = CoxPHFitter()
    cph.fit(surv_df[["time", "event", "adi_natrank", "age", "baseline_hba1c"]],
            duration_col="time", event_col="event")
    print(cph.summary)
else:
    print("lifelines not installed; skipping Cox PH section.")


In [ ]:
##interpretability

# %% [markdown]
# # 11. Interpretability: Tree Models + SHAP + Equity by ADI
#
# In this section we:
# - Fit XGBoost models for 6m / 12m / 24m horizons on the sampled, weighted cohorts
# - Compute global feature importance
# - Use SHAP to explain predictions and visualize the role of ADI
# - Evaluate AUC and calibration by ADI quintile (equity/fairness slice)

# %%
# Install if needed:
# pip install xgboost shap

from xgboost import XGBClassifier
import shap
from sklearn.metrics import roc_auc_score, brier_score_loss
from sklearn.calibration import calibration_curve

shap.initjs()  # for notebook JS visuals (optional)

# %% [markdown]
# ## 11.1 Helper: Prepare Feature Matrix and Weights

# %%
FEATURES_BASE = [
    "adi_natrank",
    "age",
    "baseline_hba1c",
    "hba1c_slope",
    "hf",
    "ckd",
    "stroke",
    "athscltc",
    "stemi"
]

def prepare_features(df_in, outcome_col):
    """
    Prepare X, y, weights for XGBoost.
    - Adds sex_female feature.
    - Uses the same baseline features as the GLM.
    """
    df = df_in.copy()

    # Sex indicator
    df["sex_female"] = np.where(df["sex"].astype(str).str.upper().str.startswith("F"), 1, 0)

    feature_cols = [f for f in FEATURES_BASE if f in df.columns] + ["sex_female"]

    X = df[feature_cols].copy()
    X = X.fillna(X.median())

    y = df[outcome_col].astype(int)

    # Final weight = design * IPW (already created earlier)
    if "final_weight" in df.columns:
        w = df["final_weight"].values
    else:
        w = np.ones(len(df))

    return X, y, w, feature_cols

X6,  y6,  w6,  feat6  = prepare_features(cohort6_small,  "out_6mo")
X12, y12, w12, feat12 = prepare_features(cohort12_small, "out_1yr")
X24, y24, w24, feat24 = prepare_features(cohort24_small, "out_2yr")

# %% [markdown]
# ## 11.2 Fit XGBoost Models (One per Horizon)
#
# We use a modest tree depth and regularization; this is a template,
# you can tune hyperparameters as needed.

# %%
def fit_xgb(X, y, sample_weight):
    model = XGBClassifier(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        random_state=42,
        eval_metric="logloss",
        n_jobs=-1
    )
    model.fit(X, y, sample_weight=sample_weight)
    return model

xgb6  = fit_xgb(X6,  y6,  w6)
xgb12 = fit_xgb(X12, y12, w12)
xgb24 = fit_xgb(X24, y24, w24)

# %% [markdown]
# ## 11.3 Global Feature Importance (Gain)
#
# This is a quick, high-level interpretability view.

# %%
def plot_feature_importance(model, feature_names, title):
    importance = model.get_booster().get_score(importance_type="gain")
    # importance is a dict keyed by 'f0', 'f1', ...
    imp_items = []
    for k, v in importance.items():
        idx = int(k[1:])   # 'f0' -> 0
        fname = feature_names[idx]
        imp_items.append((fname, v))
    imp_df = pd.DataFrame(imp_items, columns=["feature", "gain"]).sort_values("gain", ascending=False)

    print(f"\n{title}")
    print(imp_df)

    plt.figure(figsize=(6, 4))
    plt.barh(imp_df["feature"], imp_df["gain"])
    plt.gca().invert_yaxis()
    plt.title(title)
    plt.xlabel("Gain")
    plt.tight_layout()
    plt.show()

plot_feature_importance(xgb6,  feat6,  "XGBoost Feature Importance (6-month)")
plot_feature_importance(xgb12, feat12, "XGBoost Feature Importance (12-month)")
plot_feature_importance(xgb24, feat24, "XGBoost Feature Importance (24-month)")

# %% [markdown]
# ## 11.4 SHAP: Global Summary & ADI Dependence
#
# SHAP gives more nuanced explanations:
# - Summary plot: which features drive predictions most
# - Dependence plot for ADI: how risk changes with ADI, controlling for other features

# %%
def shap_analysis(model, X, feature_names, title_prefix=""):
    # SHAP TreeExplainer
    explainer = shap.TreeExplainer(model)
    shap_vals = explainer.shap_values(X)

    # Summary plot (global importance)
    shap.summary_plot(
        shap_vals,
        X,
        feature_names=feature_names,
        show=True
    )

    # ADI dependence, if present
    if "adi_natrank" in feature_names:
        shap.dependence_plot(
            "adi_natrank",
            shap_vals,
            X,
            feature_names=feature_names,
            interaction_index=None,
            show=True
        )

# 6-month SHAP
shap_analysis(xgb6,  X6,  feat6,  title_prefix="6m")

# 12-month SHAP
shap_analysis(xgb12, X12, feat12, title_prefix="12m")

# 24-month SHAP
shap_analysis(xgb24, X24, feat24, title_prefix="24m")

# %% [markdown]
# ## 11.5 Group-wise Performance & Calibration by ADI Quintile
#
# To evaluate equity, we:
# - Compute AUC by ADI quintile
# - Plot calibration curves stratified by ADI quintile

# %%
from sklearn.metrics import roc_auc_score

def group_auc(df_in, model, X, y, group_col, feature_names, label):
    df_tmp = df_in.copy()
    df_tmp = df_tmp.reset_index(drop=True)

    # Get predicted probabilities aligned to df_tmp
    y_prob = model.predict_proba(X)[:, 1]
    df_tmp["y_true"] = y
    df_tmp["y_prob"] = y_prob

    print(f"\nAUC by {group_col} for {label}")
    for g, gdf in df_tmp.groupby(group_col):
        if gdf["y_true"].nunique() < 2:
            continue
        auc = roc_auc_score(gdf["y_true"], gdf["y_prob"])
        print(f"  Group {g}: N={len(gdf)}, AUC={auc:.3f}")

def group_calibration(df_in, model, X, y, group_col, label, n_bins=10):
    df_tmp = df_in.copy().reset_index(drop=True)
    y_prob = model.predict_proba(X)[:, 1]
    df_tmp["y_true"] = y
    df_tmp["y_prob"] = y_prob

    groups = sorted(df_tmp[group_col].dropna().unique())
    plt.figure(figsize=(6, 4))

    for g in groups:
        gdf = df_tmp[df_tmp[group_col] == g]
        if len(gdf) < 50:  # skip very small groups
            continue
        frac_pos, mean_pred = calibration_curve(
            gdf["y_true"], gdf["y_prob"], n_bins=n_bins, strategy="quantile"
        )
        plt.plot(mean_pred, frac_pos, marker="o", label=f"{group_col}={g}")

    plt.plot([0, 1], [0, 1], "--")
    plt.xlabel("Predicted probability")
    plt.ylabel("Observed fraction uncontrolled")
    plt.title(f"Calibration by {group_col} ({label})")
    plt.legend()
    plt.tight_layout()
    plt.show()

# For group-level analysis we need the cohorts with their ADI quintile and sample/weights.
# We'll re-use the small cohorts and the X matrices (they align row-wise).

# Add adi_quintile to the small cohorts if not present (it should be from baseline).
for c in [cohort6_small, cohort12_small, cohort24_small]:
    c["adi_quintile"] = c["adi_quintile"].astype(int)

# 6-month: AUC & calibration by ADI quintile
group_auc(cohort6_small, xgb6, X6, y6, "adi_quintile", feat6, label="6m")
group_calibration(cohort6_small, xgb6, X6, y6, "adi_quintile", label="6m")

# 12-month:
group_auc(cohort12_small, xgb12, X12, y12, "adi_quintile", feat12, label="12m")
group_calibration(cohort12_small, xgb12, X12, y12, "adi_quintile", label="12m")

# 24-month:
group_auc(cohort24_small, xgb24, X24, y24, "adi_quintile", feat24, label="24m")
group_calibration(cohort24_small, xgb24, X24, y24, "adi_quintile", label="24m")
